# 03_feature_engineering_kfold_from_manifest

This notebook builds a classical ML pipeline for **AI-generated vs authentic image detection** using **engineered image features** rather than deep end-to-end fine-tuning.

It follows the same overall structure as the ResNet / CLIP notebooks:

1. Config  
2. Imports and helpers  
3. Load manifests  
4. Feature engineering  
5. Save engineered features  
6. K-Fold training and OOF evaluation  
7. Threshold tuning  
8. Save outputs  
9. Build submission files

## Feature families included

- Basic RGB statistics
- HSV statistics
- Grayscale statistics
- Sharpness / Laplacian variance
- Edge density
- JPEG file size
- FFT / frequency-domain features
- LBP texture histogram
- GLCM texture statistics

In [5]:
# =========================
# 1. Config
# =========================

from pathlib import Path
import torch

CONFIG = {
    # --- project / files ---
    "project_root_override": None,
    "root_markers": ["data2", "outputs"],

    "train_manifest_relpath": "outputs/manifests/train_manifest_etsy_only.csv",
    "test_manifest_relpath": "outputs/manifests/test_manifest_etsy.csv",

    "experiment_name": "features_rf_etsy_only_v1",

    # --- labels ---
    "positive_label": "ai_generated",
    "negative_label": "authentic",

    # --- feature engineering ---
    "feature_cache_dir_relpath": "outputs/features",
    "rebuild_features": False,
    "sample_size": None,   # e.g. 500 for debugging

    # --- CV / training ---
    "random_state": 42,
    "n_splits": 5,

    # --- classifier ---
    # Options: "rf", "xgb" (if xgboost installed), "logreg", "knn"
    "classifier_type": "rf",

    "rf_n_estimators": 400,
    "rf_max_depth": None,
    "rf_min_samples_leaf": 1,
    "rf_n_jobs": -1,
    "use_class_weights": True,

    "knn_n_neighbors": 7,
    "knn_weights": "distance",
    "knn_metric": "euclidean",

    "logreg_max_iter": 4000,

    # --- outputs ---
    "save_oof_csv": True,
    "save_test_probs_csv": True,
    "save_submission_csv": True,
}

In [6]:
# =========================
# 2. Imports and helpers
# =========================

import json
import math
import warnings

import numpy as np
import pandas as pd

from PIL import Image, UnidentifiedImageError
from IPython.display import display

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    import cv2
except Exception as e:
    raise ImportError("OpenCV is required for this notebook. Install with: pip install opencv-python") from e

try:
    from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
except Exception as e:
    raise ImportError(
        "scikit-image is required for this notebook. Install with: pip install scikit-image"
    ) from e

from pandas.api.types import is_numeric_dtype

warnings.filterwarnings("ignore")


def find_project_root(start: Path, markers: list[str]) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if all((candidate / m).exists() for m in markers):
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. "
        f"Expected markers: {markers}"
    )


def _resolve_path(project_root: Path, p: str | Path) -> Path:
    p = Path(p).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (project_root / p).resolve()


def safe_open_rgb(path: str | Path) -> Image.Image:
    return Image.open(path).convert("RGB")


def build_target_column(label_series: pd.Series, label2id: dict) -> pd.Series:
    if is_numeric_dtype(label_series):
        return label_series.astype(int)

    normalized = label_series.astype(str).str.strip()
    target = normalized.map(label2id)

    if target.isna().any():
        bad = sorted(normalized[target.isna()].unique())
        raise ValueError(f"Unmapped labels: {bad}")

    return target.astype(int)

In [27]:
# =========================
# 3. Locate project and load manifests
# =========================

cwd = Path.cwd()

if CONFIG["project_root_override"]:
    PROJECT_ROOT = Path(CONFIG["project_root_override"]).expanduser().resolve()
else:
    PROJECT_ROOT = find_project_root(cwd, CONFIG["root_markers"])

TRAIN_MANIFEST = _resolve_path(PROJECT_ROOT, CONFIG["train_manifest_relpath"])
TEST_MANIFEST = _resolve_path(PROJECT_ROOT, CONFIG["test_manifest_relpath"])



OUTPUT_DIR = PROJECT_ROOT / "outputs" / CONFIG["experiment_name"]

EXPERIMENT_DIR = PROJECT_ROOT / "outputs" / "experiments" / CONFIG["experiment_name"]
MODELS_DIR = EXPERIMENT_DIR / "models"
REPORTS_DIR = EXPERIMENT_DIR / "reports"
SUBMISSIONS_DIR = EXPERIMENT_DIR / "submissions"
CACHE_DIR = EXPERIMENT_DIR / "cache"


FEATURE_CACHE_DIR = CACHE_DIR / "features"

for d in [OUTPUT_DIR, FEATURE_CACHE_DIR, EXPERIMENT_DIR, MODELS_DIR, REPORTS_DIR, SUBMISSIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("OUTPUT_DIR     :", OUTPUT_DIR)
print("EXPERIMENT_DIR :", EXPERIMENT_DIR)
print("TRAIN_MANIFEST :", TRAIN_MANIFEST)
print("TEST_MANIFEST  :", TEST_MANIFEST)
print("FEATURE_CACHE  :", FEATURE_CACHE_DIR)
print("REPORTS_DIR    :", REPORTS_DIR)

train_df = pd.read_csv(TRAIN_MANIFEST)
test_df = pd.read_csv(TEST_MANIFEST)

if CONFIG["sample_size"]:
    train_df = train_df.sample(n=min(CONFIG["sample_size"], len(train_df)), random_state=CONFIG["random_state"]).reset_index(drop=True)
    test_df = test_df.sample(n=min(CONFIG["sample_size"], len(test_df)), random_state=CONFIG["random_state"]).reset_index(drop=True)

label2id = {
    CONFIG["negative_label"]: 0,
    CONFIG["positive_label"]: 1,
}
id2label = {v: k for k, v in label2id.items()}

train_df["target"] = build_target_column(train_df["label"], label2id)

print("train_df shape:", train_df.shape)
print("test_df shape :", test_df.shape)
display(train_df.head())
display(test_df.head())

PROJECT_ROOT   : /home/tbeliere/Projects/DCU/AML_PJ
OUTPUT_DIR     : /home/tbeliere/Projects/DCU/AML_PJ/outputs/features_rf_etsy_only_v1
EXPERIMENT_DIR : /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1
TRAIN_MANIFEST : /home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/train_manifest_etsy_only.csv
TEST_MANIFEST  : /home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/test_manifest_etsy.csv
FEATURE_CACHE  : /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1/cache/features
REPORTS_DIR    : /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1/reports
train_df shape: (4800, 13)
test_df shape : (2058, 12)


,image_id,image_path,label,source,split,file_exists,valid_image,width,height,mode,format,aspect_ratio,target
0,000541e8-6ea3-4ae0-a79b-10bd3bc3a38d.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000,1
1,00329fac-20e5-41f7-92df-05b256fdf9e8.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000,1
2,005af84d-d9ae-4842-a4a2-75fc8f590a94.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000,1
3,0068b91f-794a-4520-83a4-0f8735d2a229.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,896,1195,RGB,JPEG,0.749791,1
4,009ad604-af12-4c5d-b6a6-81da359c13a1.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,558,RGB,JPEG,1.835125,1


,image_id,image_path,label,source,split,file_exists,valid_image,width,height,mode,format,aspect_ratio
0,000317ef-6928-48ca-9b51-7742ab8dfb86.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,NaN,etsy,test,True,True,1024,1024,RGB,JPEG,1.000000
1,000a811d-cf8f-4b5b-be75-bd9f1a73c098.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,NaN,etsy,test,True,True,1024,1024,RGB,JPEG,1.000000
2,003a1394-83c5-41c0-9cd5-18ac89397c7b.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,NaN,etsy,test,True,True,1195,896,RGB,JPEG,1.333705
3,00888016-16c4-4ce7-a03e-073f04a1a9c8.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,NaN,etsy,test,True,True,1024,1024,RGB,JPEG,1.000000
4,00e301e1-7520-4abb-8007-9e00a0b04da6.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,NaN,etsy,test,True,True,896,1195,RGB,JPEG,0.749791


In [14]:
# =========================
# 4. Feature engineering
# =========================

def _rgb_stats(rgb: np.ndarray) -> dict:
    feats = {}
    for i, c in enumerate(["r", "g", "b"]):
        channel = rgb[:, :, i].astype(np.float32)
        feats[f"{c}_mean"] = float(channel.mean())
        feats[f"{c}_std"] = float(channel.std())
        feats[f"{c}_min"] = float(channel.min())
        feats[f"{c}_max"] = float(channel.max())
    return feats


def _hsv_stats(rgb: np.ndarray) -> dict:
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    feats = {}
    for i, c in enumerate(["h", "s", "v"]):
        channel = hsv[:, :, i].astype(np.float32)
        feats[f"{c}_mean"] = float(channel.mean())
        feats[f"{c}_std"] = float(channel.std())
    return feats


def _gray_stats(gray: np.ndarray) -> dict:
    grayf = gray.astype(np.float32)
    return {
        "gray_mean": float(grayf.mean()),
        "gray_std": float(grayf.std()),
        "gray_p10": float(np.percentile(grayf, 10)),
        "gray_p50": float(np.percentile(grayf, 50)),
        "gray_p90": float(np.percentile(grayf, 90)),
    }


def _sharpness_and_edges(gray: np.ndarray) -> dict:
    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    edges = cv2.Canny(gray, 100, 200)
    edge_density = float((edges > 0).mean())
    grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    return {
        "laplacian_var": lap_var,
        "edge_density": edge_density,
        "grad_mag_mean": float(grad_mag.mean()),
        "grad_mag_std": float(grad_mag.std()),
    }


def _fft_features(gray: np.ndarray) -> dict:
    grayf = gray.astype(np.float32)
    fft = np.fft.fft2(grayf)
    fft_shift = np.fft.fftshift(fft)
    mag = np.log1p(np.abs(fft_shift))

    h, w = gray.shape
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    radius = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    rmax = radius.max() + 1e-8

    low_mask = radius <= 0.15 * rmax
    mid_mask = (radius > 0.15 * rmax) & (radius <= 0.4 * rmax)
    high_mask = radius > 0.4 * rmax

    total = mag.mean() + 1e-8
    low = float(mag[low_mask].mean())
    mid = float(mag[mid_mask].mean())
    high = float(mag[high_mask].mean())

    return {
        "fft_mean": float(mag.mean()),
        "fft_std": float(mag.std()),
        "fft_low_mean": low,
        "fft_mid_mean": mid,
        "fft_high_mean": high,
        "fft_high_over_total": float(high / total),
        "fft_low_over_total": float(low / total),
    }


def _lbp_features(gray: np.ndarray, n_bins: int = 10) -> dict:
    radius = 1
    n_points = 8 * radius
    lbp = local_binary_pattern(gray, P=n_points, R=radius, method="uniform")
    hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, n_points + 3),
        range=(0, n_points + 2),
        density=True,
    )
    feats = {}
    for i, v in enumerate(hist[:n_bins]):
        feats[f"lbp_hist_{i}"] = float(v)
    feats["lbp_entropy"] = float(-(hist * np.log(hist + 1e-12)).sum())
    return feats


def _glcm_features(gray: np.ndarray) -> dict:
    # Reduce to 32 gray levels for stability/speed
    gray_q = (gray / 8).astype(np.uint8)
    glcm = graycomatrix(gray_q, distances=[1], angles=[0, np.pi/4, np.pi/2], levels=32, symmetric=True, normed=True)

    feats = {}
    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        vals = graycoprops(glcm, prop)
        feats[f"glcm_{prop}_mean"] = float(vals.mean())
        feats[f"glcm_{prop}_std"] = float(vals.std())
    return feats


def extract_engineered_features(image_path: str | Path) -> dict:
    image_path = Path(image_path)

    with safe_open_rgb(image_path) as img:
        rgb = np.array(img)

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    feats = {
        "image_path": str(image_path),
        "width": int(rgb.shape[1]),
        "height": int(rgb.shape[0]),
        "aspect_ratio": float(rgb.shape[1] / max(rgb.shape[0], 1)),
        "file_size_bytes": int(image_path.stat().st_size),
    }

    feats.update(_rgb_stats(rgb))
    feats.update(_hsv_stats(rgb))
    feats.update(_gray_stats(gray))
    feats.update(_sharpness_and_edges(gray))
    feats.update(_fft_features(gray))
    feats.update(_lbp_features(gray))
    feats.update(_glcm_features(gray))

    return feats


def build_or_load_feature_table(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    cache_path = FEATURE_CACHE_DIR / f"{split_name}_engineered_features.csv"

    if cache_path.exists() and not CONFIG["rebuild_features"]:
        print(f"[CACHE] Loading {split_name} features from {cache_path}")
        return pd.read_csv(cache_path)

    rows = []
    for _, row in df.iterrows():
        feats = extract_engineered_features(row["image_path"])
        feats["image_id"] = row["image_id"]
        if "label" in row.index:
            feats["label"] = row.get("label", pd.NA)
        if "target" in row.index:
            feats["target"] = row.get("target", pd.NA)
        rows.append(feats)

    feat_df = pd.DataFrame(rows)
    feat_df.to_csv(cache_path, index=False)
    print(f"[SAVE] Wrote {split_name} features to {cache_path}")
    return feat_df

In [15]:
# =========================
# 5. Build and save engineered features
# =========================

train_feat_df = build_or_load_feature_table(train_df, "train")
test_feat_df = build_or_load_feature_table(test_df, "test")

print("train_feat_df shape:", train_feat_df.shape)
print("test_feat_df shape :", test_feat_df.shape)
display(train_feat_df.head())
display(test_feat_df.head())

[SAVE] Wrote train features to /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1/cache/features/train_engineered_features.csv
[SAVE] Wrote test features to /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1/cache/features/test_engineered_features.csv
train_feat_df shape: (4800, 65)
test_feat_df shape : (2058, 64)


,image_path,width,height,aspect_ratio,file_size_bytes,r_mean,r_std,r_min,r_max,g_mean,...,glcm_homogeneity_std,glcm_energy_mean,glcm_energy_std,glcm_correlation_mean,glcm_correlation_std,glcm_ASM_mean,glcm_ASM_std,image_id,label,target
0,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,1024,1024,1.000000,294510,92.909805,61.088974,0.0,255.0,79.731331,...,0.021669,0.156624,0.004629,0.963123,0.008861,0.024552,0.001436,000541e8-6ea3-4ae0-a79b-10bd3bc3a38d.jpg,ai_generated,1
1,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,1024,1024,1.000000,173762,185.961609,65.416901,0.0,255.0,153.508957,...,0.013228,0.340400,0.001701,0.994725,0.001013,0.115875,0.001156,00329fac-20e5-41f7-92df-05b256fdf9e8.jpg,ai_generated,1
2,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,1024,1024,1.000000,312224,198.543549,56.145897,0.0,255.0,162.262817,...,0.025533,0.221401,0.002393,0.981482,0.004753,0.049024,0.001056,005af84d-d9ae-4842-a4a2-75fc8f590a94.jpg,ai_generated,1
3,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,896,1195,0.749791,228814,158.416321,44.203621,0.0,255.0,143.913315,...,0.039576,0.227368,0.007454,0.940037,0.015498,0.051752,0.003403,0068b91f-794a-4520-83a4-0f8735d2a229.jpg,ai_generated,1
4,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,1024,558,1.835125,73476,194.542007,42.908024,0.0,255.0,172.663635,...,0.023460,0.339746,0.005747,0.979239,0.006285,0.115460,0.003885,009ad604-af12-4c5d-b6a6-81da359c13a1.jpg,ai_generated,1


,image_path,width,height,aspect_ratio,file_size_bytes,r_mean,r_std,r_min,r_max,g_mean,...,glcm_homogeneity_mean,glcm_homogeneity_std,glcm_energy_mean,glcm_energy_std,glcm_correlation_mean,glcm_correlation_std,glcm_ASM_mean,glcm_ASM_std,image_id,label
0,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,1024,1024,1.000000,243364,79.550392,108.894547,0.0,255.0,76.517731,...,0.810895,0.029160,0.481751,0.010316,0.974427,0.008380,0.232190,0.009878,000317ef-6928-48ca-9b51-7742ab8dfb86.jpg,NaN
1,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,1024,1024,1.000000,257952,190.052139,61.423672,0.0,255.0,177.330994,...,0.741836,0.028543,0.168879,0.005999,0.944862,0.011319,0.028556,0.002009,000a811d-cf8f-4b5b-be75-bd9f1a73c098.jpg,NaN
2,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,1195,896,1.333705,66638,136.058334,79.317604,23.0,242.0,137.707809,...,0.960332,0.009013,0.402823,0.003341,0.998670,0.000419,0.162277,0.002685,003a1394-83c5-41c0-9cd5-18ac89397c7b.jpg,NaN
3,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,1024,1024,1.000000,192024,218.554321,44.572155,0.0,255.0,213.062714,...,0.834140,0.023285,0.361146,0.015671,0.937954,0.018899,0.130672,0.011399,00888016-16c4-4ce7-a03e-073f04a1a9c8.jpg,NaN
4,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,896,1195,0.749791,410524,170.266342,38.370968,0.0,255.0,166.733887,...,0.540047,0.001054,0.128084,0.000901,0.914599,0.007335,0.016406,0.000230,00e301e1-7520-4abb-8007-9e00a0b04da6.jpg,NaN


In [16]:
# =========================
# 6. Prepare model matrix
# =========================

non_feature_cols = {"image_id", "image_path", "label", "target"}
feature_cols = [c for c in train_feat_df.columns if c not in non_feature_cols]

X_train = train_feat_df[feature_cols].copy()
X_test = test_feat_df[feature_cols].copy()
y = train_feat_df["target"].astype(int).values

print("Number of engineered features:", len(feature_cols))
print("First 20 feature columns:", feature_cols[:20])
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

Number of engineered features: 61
First 20 feature columns: ['width', 'height', 'aspect_ratio', 'file_size_bytes', 'r_mean', 'r_std', 'r_min', 'r_max', 'g_mean', 'g_std', 'g_min', 'g_max', 'b_mean', 'b_std', 'b_min', 'b_max', 'h_mean', 'h_std', 's_mean', 's_std']
X_train shape: (4800, 61)
X_test shape : (2058, 61)


In [17]:
# =========================
# 7. Classifier factory
# =========================

def make_classifier():
    clf_type = CONFIG["classifier_type"]

    if clf_type == "rf":
        class_weight = "balanced" if CONFIG["use_class_weights"] else None
        return RandomForestClassifier(
            n_estimators=CONFIG["rf_n_estimators"],
            max_depth=CONFIG["rf_max_depth"],
            min_samples_leaf=CONFIG["rf_min_samples_leaf"],
            n_jobs=CONFIG["rf_n_jobs"],
            random_state=CONFIG["random_state"],
            class_weight=class_weight,
        )

    if clf_type == "knn":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(
                n_neighbors=CONFIG["knn_n_neighbors"],
                weights=CONFIG["knn_weights"],
                metric=CONFIG["knn_metric"],
            ))
        ])

    if clf_type == "logreg":
        class_weight = "balanced" if CONFIG["use_class_weights"] else None
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=CONFIG["logreg_max_iter"],
                class_weight=class_weight,
                random_state=CONFIG["random_state"],
            ))
        ])

    if clf_type == "xgb":
        if not HAS_XGBOOST:
            raise ImportError("xgboost is not installed. Install with: pip install xgboost")
        return XGBClassifier(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=CONFIG["random_state"],
            n_jobs=-1,
        )

    raise ValueError(f"Unsupported classifier_type: {clf_type}")

In [18]:
# =========================
# 8. K-Fold training
# =========================

skf = StratifiedKFold(
    n_splits=CONFIG["n_splits"],
    shuffle=True,
    random_state=CONFIG["random_state"],
)

oof_probs = np.zeros(len(train_feat_df), dtype=np.float32)
test_probs_folds = []
fold_f1s = []
best_fold_idx = None
best_fold_f1 = -1.0
best_fold_test_probs = None

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y), start=1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    clf = make_classifier()
    clf.fit(X_tr, y_tr)

    va_probs = clf.predict_proba(X_va)[:, 1]
    oof_probs[va_idx] = va_probs

    va_preds_05 = (va_probs >= 0.5).astype(int)
    fold_f1 = f1_score(y_va, va_preds_05)
    fold_f1s.append(fold_f1)

    print(f"Fold {fold} F1 @ 0.5: {fold_f1:.4f}")

    test_probs = clf.predict_proba(X_test)[:, 1]
    test_probs_folds.append(test_probs)

    if fold_f1 > best_fold_f1:
        best_fold_f1 = fold_f1
        best_fold_idx = fold
        best_fold_test_probs = test_probs.copy()

test_probs_ensemble = np.mean(test_probs_folds, axis=0)

print("\nMean fold F1 @ 0.5:", float(np.mean(fold_f1s)))
print("Std  fold F1 @ 0.5:", float(np.std(fold_f1s)))
print("Best fold:", best_fold_idx)
print("Best fold F1 @ 0.5:", float(best_fold_f1))

Fold 1 F1 @ 0.5: 0.8168
Fold 2 F1 @ 0.5: 0.8365
Fold 3 F1 @ 0.5: 0.8312
Fold 4 F1 @ 0.5: 0.8080
Fold 5 F1 @ 0.5: 0.8155

Mean fold F1 @ 0.5: 0.8215954960749766
Std  fold F1 @ 0.5: 0.010597766302283084
Best fold: 2
Best fold F1 @ 0.5: 0.8365180467091295


In [19]:
# =========================
# 9. OOF threshold tuning and evaluation
# =========================

thresholds = np.arange(0.10, 0.91, 0.01)

best_thr = 0.5
best_f1 = -1.0

for thr in thresholds:
    preds = (oof_probs >= thr).astype(int)
    score = f1_score(y, preds)
    if score > best_f1:
        best_f1 = score
        best_thr = thr

y_pred_05 = (oof_probs >= 0.5).astype(int)
y_pred_best = (oof_probs >= best_thr).astype(int)

precision_ai = precision_score(y, y_pred_best, pos_label=1)
recall_ai = recall_score(y, y_pred_best, pos_label=1)

print("OOF F1 @ 0.5     :", float(f1_score(y, y_pred_05)))
print("Best threshold   :", float(best_thr))
print("Best OOF F1      :", float(best_f1))
print("Precision (AI)   :", float(precision_ai))
print("Recall (AI)      :", float(recall_ai))

print("\nClassification report @ best threshold:")
print(classification_report(y, y_pred_best, target_names=[CONFIG["negative_label"], CONFIG["positive_label"]]))

print("Confusion matrix @ best threshold:")
print(confusion_matrix(y, y_pred_best))

OOF F1 @ 0.5     : 0.8216032319795875
Best threshold   : 0.3999999999999998
Best OOF F1      : 0.8297177961220964
Precision (AI)   : 0.7467173462335868
Recall (AI)      : 0.9334773218142549

Classification report @ best threshold:
              precision    recall  f1-score   support

   authentic       0.92      0.71      0.80      2485
ai_generated       0.75      0.93      0.83      2315

    accuracy                           0.82      4800
   macro avg       0.83      0.82      0.81      4800
weighted avg       0.84      0.82      0.81      4800

Confusion matrix @ best threshold:
[[1752  733]
 [ 154 2161]]


In [28]:
# =========================
# 10. Save OOF, test probabilities, submissions, metrics
# =========================

oof_df = pd.DataFrame({
    "image_id": train_feat_df["image_id"],
    "label": train_feat_df["label"],
    "target": train_feat_df["target"].astype(int),
    "oof_ai_generated_proba": oof_probs,
})
oof_df["oof_pred_target"] = (oof_df["oof_ai_generated_proba"] >= best_thr).astype(int)
oof_df["oof_pred_label"] = oof_df["oof_pred_target"].map(id2label)

test_probs_df = pd.DataFrame({
    "image_id": test_feat_df["image_id"],
    "fold_ensemble_ai_generated_proba": test_probs_ensemble,
    "best_fold_ai_generated_proba": best_fold_test_probs,
})

submission_best_fold = pd.DataFrame({
    "image_id": test_feat_df["image_id"],
    "ground_truth": (best_fold_test_probs >= best_thr).astype(int),
})

submission_fold_ensemble = pd.DataFrame({
    "image_id": test_feat_df["image_id"],
    "ground_truth": (test_probs_ensemble >= best_thr).astype(int),
})

metrics = {
    "f1_05": float(f1_score(y, y_pred_05)),
    "best_threshold": float(best_thr),
    "best_f1": float(best_f1),
    "precision_ai_generated": float(precision_ai),
    "recall_ai_generated": float(recall_ai),
    "fold_f1_mean_05": float(np.mean(fold_f1s)),
    "fold_f1_std_05": float(np.std(fold_f1s)),
    "best_fold": int(best_fold_idx),
    "best_fold_f1_05": float(best_fold_f1),
    "train_manifest": str(TRAIN_MANIFEST),
    "test_manifest": str(TEST_MANIFEST),
    "classifier_type": CONFIG["classifier_type"],
    "use_class_weights": bool(CONFIG["use_class_weights"]),
    "n_features": int(len(feature_cols)),
    "feature_cache_dir": str(FEATURE_CACHE_DIR),
}

if CONFIG["save_oof_csv"]:
    oof_path = REPORTS_DIR / "oof_predictions.csv"
    oof_df.to_csv(oof_path, index=False)
    print("[SAVE]", oof_path)

if CONFIG["save_test_probs_csv"]:
    test_probs_path = REPORTS_DIR / "test_probabilities.csv"
    test_probs_df.to_csv(test_probs_path, index=False)
    print("[SAVE]", test_probs_path)

if CONFIG["save_submission_csv"]:
    sub_best_path = REPORTS_DIR / "submission_best_fold.csv"
    sub_ens_path = REPORTS_DIR / "submission_fold_ensemble.csv"
    submission_best_fold.to_csv(sub_best_path, index=False)
    submission_fold_ensemble.to_csv(sub_ens_path, index=False)
    print("[SAVE]", sub_best_path)
    print("[SAVE]", sub_ens_path)

metrics_path = REPORTS_DIR / "metrics_summary.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)
print("[SAVE]", metrics_path)

display(oof_df.head())
display(test_probs_df.head())
print(json.dumps(metrics, indent=4))

[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1/reports/oof_predictions.csv
[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1/reports/test_probabilities.csv
[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1/reports/submission_best_fold.csv
[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1/reports/submission_fold_ensemble.csv
[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1/reports/metrics_summary.json


,image_id,label,target,oof_ai_generated_proba,oof_pred_target,oof_pred_label
0,000541e8-6ea3-4ae0-a79b-10bd3bc3a38d.jpg,ai_generated,1,0.6125,1,ai_generated
1,00329fac-20e5-41f7-92df-05b256fdf9e8.jpg,ai_generated,1,0.4825,1,ai_generated
2,005af84d-d9ae-4842-a4a2-75fc8f590a94.jpg,ai_generated,1,0.5025,1,ai_generated
3,0068b91f-794a-4520-83a4-0f8735d2a229.jpg,ai_generated,1,0.4225,1,ai_generated
4,009ad604-af12-4c5d-b6a6-81da359c13a1.jpg,ai_generated,1,0.8650,1,ai_generated


,image_id,fold_ensemble_ai_generated_proba,best_fold_ai_generated_proba
0,000317ef-6928-48ca-9b51-7742ab8dfb86.jpg,0.2795,0.2750
1,000a811d-cf8f-4b5b-be75-bd9f1a73c098.jpg,0.7550,0.7500
2,003a1394-83c5-41c0-9cd5-18ac89397c7b.jpg,0.4260,0.4100
3,00888016-16c4-4ce7-a03e-073f04a1a9c8.jpg,0.6935,0.7150
4,00e301e1-7520-4abb-8007-9e00a0b04da6.jpg,0.2095,0.2025


{
    "f1_05": 0.8216032319795875,
    "best_threshold": 0.3999999999999998,
    "best_f1": 0.8297177961220964,
    "precision_ai_generated": 0.7467173462335868,
    "recall_ai_generated": 0.9334773218142549,
    "fold_f1_mean_05": 0.8215954960749766,
    "fold_f1_std_05": 0.010597766302283084,
    "best_fold": 2,
    "best_fold_f1_05": 0.8365180467091295,
    "train_manifest": "/home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/train_manifest_etsy_only.csv",
    "test_manifest": "/home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/test_manifest_etsy.csv",
    "classifier_type": "rf",
    "use_class_weights": true,
    "n_features": 61,
    "feature_cache_dir": "/home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/features_rf_etsy_only_v1/cache/features"
}


In [ ]:
# =========================
# 11. Optional: feature importance (tree models only)
# =========================

if CONFIG["classifier_type"] == "rf":
    final_clf = make_classifier()
    final_clf.fit(X_train, y)

    importances = pd.DataFrame({
        "feature": feature_cols,
        "importance": final_clf.feature_importances_,
    }).sort_values("importance", ascending=False)

    importance_path = OUTPUT_DIR / "feature_importance.csv"
    importances.to_csv(importance_path, index=False)
    print("[SAVE]", importance_path)
    display(importances.head(30))
else:
    print("Feature importance export is currently implemented only for Random Forest.")